# Завдання 3. Домашня робота

###PostgreSQL installation and librarys

In [1]:
# Install PostgreSQL server from Ubuntu packages (Colab VM runs Ubuntu)
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib > /dev/null

# Start the PostgreSQL service
!service postgresql start

# Set a password for the default superuser and create a database
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE hw4_db;"

# SQLAlchemy 2.x + psycopg 3 driver (binary build, no compilation needed)
!pip install -q "sqlalchemy>=2.0" "psycopg[binary]"

from sqlalchemy import create_engine, text, URL
import pandas as pd

# Connection settings
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "hw4_db"

# URL.create escapes special characters in credentials safely
db_url = URL.create(
    drivername="postgresql+psycopg",  # dialect + DBAPI driver
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(db_url)

# # Open a connection, run a test query, close the connection automatically
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar_one()
    print(version)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
ALTER ROLE
CREATE DATABASE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 22.5 MB/s eta 0:00:00
PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


### Download data and create raw tables

In [2]:
import os
import kagglehub

# Download latest version
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)


100%|██████████| 42.6M/42.6M [00:00<00:00, 116MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### Cтворення TYPED схеми

In [3]:
# create STAGE schema
SCHEMA_SQL = text("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

""")

with engine.begin() as conn:
    conn.execute(SCHEMA_SQL)

raw_df = pd.read_sql(text("SELECT * from olist_customers LIMIT 10"), engine)
raw_df

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


### Копіювання данних з Raw у Typed

In [4]:
COPY_SQL = text("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

with engine.begin() as conn:
    conn.execute(COPY_SQL)

SMOKE = text("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")

smoke_data_df = pd.read_sql(SMOKE, engine)
smoke_data_df

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


##2. DML-запити з PostgreSQL-фічами

2.1. INSERT ... RETURNING: створення alert для продавця з найбільшою кількістю запізнень

In [5]:
SQL_CREATE_SELLER_ALERTS = text("""
        DROP TABLE IF EXISTS seller_alerts CASCADE;

        CREATE TABLE seller_alerts (
            alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
            alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
            severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
            details    TEXT,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
        """)

SQL_DMLS_LATE_DELIVERY = text("""
    WITH seller_orders AS (
        SELECT DISTINCT
            oi.seller_id,
            oi.order_id
        FROM olist_order_items AS oi
    ),
    late_sellers AS (
        SELECT
            so.seller_id,
            COUNT(DISTINCT o.order_id) AS late_orders
        FROM seller_orders AS so
        JOIN olist_orders AS o
            ON o.order_id = so.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
          AND o.order_estimated_delivery_date IS NOT NULL
          AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
        GROUP BY so.seller_id
        ORDER BY late_orders DESC, so.seller_id
        LIMIT 1
    )
    INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
    SELECT
        seller_id,
        'late_delivery',
        4,
        'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
    FROM late_sellers
    RETURNING alert_id, seller_id, alert_type, severity, created_at;
    """)

with engine.begin() as conn:
    conn.execute(SQL_CREATE_SELLER_ALERTS)
    res = conn.execute(SQL_DMLS_LATE_DELIVERY) # get results because RETURNING in query
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-10-21 17:52:05.047066+00:00


2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert агрегованих seller-score

In [6]:
SQL_CREATE = text("""
            DROP TABLE IF EXISTS seller_score CASCADE;

            CREATE TABLE seller_score (
                seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
                avg_rating  NUMERIC(3, 2),
                n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
                tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
                updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
            );
            """)

SQL_UPSERT = text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),
        review_per_order AS (
            SELECT
                rv.order_id,
                AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
            FROM olist_order_reviews AS rv
            WHERE rv.review_score IS NOT NULL
            GROUP BY rv.order_id
        ),
        seller_rating AS (
            SELECT
                so.seller_id,
                AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
                COUNT(rpo.order_id) AS n_reviews
            FROM seller_orders AS so
            LEFT JOIN review_per_order AS rpo
                ON rpo.order_id = so.order_id
            GROUP BY so.seller_id
        )
        INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
        SELECT
            seller_id,
            avg_rating,
            n_reviews,
            CASE
                WHEN avg_rating IS NULL THEN NULL
                WHEN avg_rating >= 4.5 THEN 'gold'
                WHEN avg_rating >= 3.5 THEN 'silver'
                ELSE 'bronze'
            END AS tier
        FROM seller_rating
        ON CONFLICT (seller_id) DO UPDATE
        SET avg_rating = EXCLUDED.avg_rating,
            n_reviews  = EXCLUDED.n_reviews,
            tier       = EXCLUDED.tier,
            updated_at = NOW()
        RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
        """)

with engine.begin() as conn:
    conn.execute(SQL_CREATE)
    res = conn.execute(SQL_UPSERT)
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-21 17:52:05.239443+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-21 17:52:05.239443+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-21 17:52:05.239443+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-21 17:52:05.239443+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-21 17:52:05.239443+00:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-10-21 17:52:05.239443+00:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-10-21 17:52:05.239443+00:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-10-21 17:52:05.239443+00:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-10-21 17:52:05.239443+00:00


## Пояснення запиту `seller_score`

**1. `SELECT DISTINCT` у `seller_orders`**

`olist_order_items` має один рядок на товар. Замовлення з 3 товарами одного продавця дало б 3 однакові пари `(seller_id, order_id)`, і review цього замовлення порахувався б 3 рази. `DISTINCT` залишає одну пару "продавець - замовлення".

**2. Агрегація review по `order_id`**

Одне замовлення може мати кілька review. Без агрегації `JOIN` розмножив би рядки (fan-out - розмноження рядків під час `JOIN`), і такі замовлення мали б більшу вагу. `GROUP BY order_id` дає одну оцінку на замовлення.

**3. `ON CONFLICT (seller_id) DO UPDATE`**

`seller_id` - унікальний ключ. Якщо продавця ще немає - рядок вставляється, якщо вже є - оновлюється значеннями з `EXCLUDED` (рядок, який намагалися вставити). Тому повторний запуск не створює дублікатів, а лише оновлює дані.

**4. Продавці без review**

- `LEFT JOIN` зберігає продавця, але поля review = `NULL`.
- `AVG` по самих `NULL` повертає `NULL` → `avg_rating = NULL`.
- `COUNT(rpo.order_id)` рахує лише не-`NULL` → `n_reviews = 0`.
- `CASE WHEN avg_rating IS NULL THEN NULL` → `tier = NULL`. Без цієї умови продавець потрапив би в `ELSE 'bronze'`.

2.3. UPDATE ... FROM: оновлення прапорця запізнення в olist_orders

In [7]:
SQL_ALTER = text("""
        ALTER TABLE olist_orders
        ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
    """)

SQL_UPDATE = text("""
    WITH delivery_flags AS (
        SELECT
            order_id,
            (
                order_delivered_customer_date IS NOT NULL
                AND order_estimated_delivery_date IS NOT NULL
                AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
            ) AS is_late_calc
        FROM olist_orders
    )
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
    RETURNING o.order_id, o.customer_id, o.is_late;
""")

with engine.begin() as conn:
    conn.execute(SQL_ALTER)
    res = conn.execute(SQL_UPDATE)
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,False
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,False
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,False
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,False


2.4. DELETE ... RETURNING: контрольоване видалення застарілих alert-ів

Якщо результат порожній — це означає що за останні 30 днів немає застарілих алертів (немає записів яки були створені більш ніж 30 днів тому). `WHERE created_at < NOW() - INTERVAL '30 days'`

In [8]:
SQL_DELETE = text("""
        DELETE FROM seller_alerts
        WHERE created_at < NOW() - INTERVAL '30 days'
        RETURNING alert_id, seller_id, alert_type, created_at;
    """)

with engine.begin() as conn:
    res = conn.execute(SQL_DELETE)
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,alert_id,seller_id,alert_type,created_at


### 3. CREATE TABLE customer_segments

In [9]:
SQL_CREATE = text("""
    DROP TABLE IF EXISTS olist_customer_dim CASCADE;

    CREATE TABLE olist_customer_dim (
        customer_unique_id TEXT PRIMARY KEY,
        latest_customer_id TEXT REFERENCES olist_customers(customer_id),
        customer_state     CHAR(2),
        customer_city      TEXT
    );
""")

SQL_INSERT = text("""
    INSERT INTO olist_customer_dim (
        customer_unique_id,
        latest_customer_id,
        customer_state,
        customer_city
    )
    SELECT DISTINCT ON (c.customer_unique_id)
        c.customer_unique_id,
        c.customer_id AS latest_customer_id,
        c.customer_state,
        c.customer_city
    FROM olist_customers AS c
    LEFT JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    ORDER BY
        c.customer_unique_id,
        o.order_purchase_timestamp DESC NULLS LAST,
        c.customer_id;
""")

with engine.begin() as conn:
    conn.execute(SQL_CREATE)
    conn.execute(SQL_INSERT)
res = pd.read_sql("SELECT * FROM olist_customer_dim LIMIT 10", engine)
display(res)

,customer_unique_id,latest_customer_id,customer_state,customer_city
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,SP,cajamar
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,SP,osasco
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,SC,sao jose
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,PA,belem
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,SP,sorocaba
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,SP,sao paulo
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,SP,campinas
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,PR,curitiba
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,RJ,teresopolis
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,MA,sao luis


customer_segments creation

In [10]:
SQL_CREATE = text("""
DROP TABLE IF EXISTS customer_segments CASCADE;
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);
        """)

SQL_ALTER = text("""
ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

SQL_CHECK = text("""
SELECT EXISTS (
    SELECT 1
    FROM information_schema.tables
    WHERE table_name = 'customer_segments'
) as table_exists
""")

with engine.begin() as conn:
    conn.execute(SQL_CREATE)
    conn.execute(SQL_ALTER)
    res = conn.execute(SQL_CHECK)
res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,table_exists
0,True


In [11]:
FILL_QUERY = text("""
    WITH reference_date AS (
        SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),
    order_totals AS (
        SELECT
            o.order_id,
            o.customer_id,
            o.order_purchase_timestamp::DATE AS order_date,
            COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            o.customer_id,
            o.order_purchase_timestamp
    ),
    customer_stats AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT ot.order_id) AS n_orders,
            COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
            MAX(ot.order_date) AS last_order_date
        FROM olist_customer_dim AS cd
        JOIN olist_customers AS c
            ON c.customer_unique_id = cd.customer_unique_id
        LEFT JOIN order_totals AS ot
            ON ot.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    ),
    segmented AS (
        SELECT
            cs.customer_unique_id,
            cs.n_orders,
            cs.monetary_value,
            CASE
                WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
            END AS recency_days,
            CASE
                WHEN cs.monetary_value >= 1000 THEN 'VIP'
                WHEN cs.last_order_date IS NOT NULL
                      AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                      AND cs.n_orders = 1 THEN 'new'
                WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
                ELSE 'inactive'
            END AS segment_name,
            CASE
                WHEN cs.monetary_value >= 1000 THEN 5
                WHEN cs.monetary_value >= 500  THEN 4
                WHEN cs.n_orders >= 2          THEN 3
                WHEN cs.n_orders = 1           THEN 2
                ELSE 1
            END::SMALLINT AS rfm_score
        FROM customer_stats AS cs
        CROSS JOIN reference_date AS rd
    )
    INSERT INTO customer_segments (
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    )
    SELECT
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders
    FROM segmented
    ON CONFLICT (customer_unique_id) DO UPDATE
    SET segment_name   = EXCLUDED.segment_name,
        rfm_score      = EXCLUDED.rfm_score,
        recency_days   = EXCLUDED.recency_days,
        monetary_value = EXCLUDED.monetary_value,
        n_orders       = EXCLUDED.n_orders,
        assigned_at    = NOW()
    RETURNING
        customer_unique_id,
        segment_name,
        rfm_score,
        recency_days,
        monetary_value,
        n_orders;
""")

with engine.begin() as conn:
    res_fill = conn.execute(FILL_QUERY)
res_fill_df = pd.DataFrame(res_fill.fetchall(), columns=list(res_fill.keys()))
display(res_fill_df)

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1


Check constrains

In [12]:
SQL_CHECK = text("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

with engine.begin() as conn:
    res_check = conn.execute(SQL_CHECK)
res_check_df = pd.DataFrame(res_check.fetchall(), columns=list(res_check.keys()))
display(res_check_df)

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


**1. Чому `customer_unique_id`**

`customer_id` в Olist унікальний для кожного замовлення, а `customer_unique_id` - для людини. Сегментація має враховувати всі замовлення клієнта.

**2. Навіщо `olist_customer_dim`**

FK може посилатися лише на `PRIMARY KEY` або `UNIQUE`. В `olist_customers` `customer_unique_id` повторюється, а в `olist_customer_dim` він `PRIMARY KEY`.

**3. Чому `reference_date` з датасету**

Дані закінчуються у 2018 році. З `CURRENT_DATE` усі клієнти мали б тисячі днів `recency_days`, і результат змінювався б щодня.

**4. Як працює `ON CONFLICT`**

`UNIQUE (customer_unique_id)` дає один рядок на клієнта. При повторному запуску рядок оновлюється, а не дублюється.

### 4. JOIN-запити з business-interpretation

In [14]:
SQL_TOP_REV = text("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

SQL_CUST_LEVEL_FTRS = text("""
    WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

SQL_STATES = text("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

SQL_SAME_STATE_DIFF_CITY = text("""
    SELECT
        s1.seller_id AS seller_a,
        s2.seller_id AS seller_b,
        s1.seller_state,
        s1.seller_city AS city_a,
        s2.seller_city AS city_b
    FROM olist_sellers AS s1
    JOIN olist_sellers AS s2
        ON s1.seller_state = s2.seller_state
        AND s1.seller_id < s2.seller_id
        AND s1.seller_city IS DISTINCT FROM s2.seller_city
    ORDER BY s1.seller_state, seller_a, seller_b
    LIMIT 20;
""")

SQL_NO_FEEDBACK_ORDER = text("""
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM olist_orders AS o
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = o.order_id
    WHERE rv.review_row_id IS NULL
    ORDER BY o.order_purchase_timestamp
    LIMIT 20;
""")

queries = [
    {'title': '4.1. INNER JOIN: топ категорій за revenue і seller-state', 'stmt': SQL_TOP_REV},
    {'title': '4.2. LEFT JOIN + COALESCE: customer-level features без множення review', 'stmt': SQL_CUST_LEVEL_FTRS},
    {'title': '4.3. FULL OUTER JOIN: які штати присутні серед клієнтів і продавців', 'stmt': SQL_STATES},
    {'title': '4.4. SELF JOIN: пари продавців з одного штату, але з різних міст', 'stmt': SQL_SAME_STATE_DIFF_CITY},
    {'title': '4.5. Anti-join: замовлення без відгуку', 'stmt': SQL_NO_FEEDBACK_ORDER},
]

for query in queries:
    with engine.begin() as conn:
        res = conn.execute(query['stmt'])
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
    print("-"*100)
    print(query['title'])
    print("-"*100)
    display(res_df)

----------------------------------------------------------------------------------------------------
4.1. INNER JOIN: топ категорій за revenue і seller-state
----------------------------------------------------------------------------------------------------


,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


----------------------------------------------------------------------------------------------------
4.2. LEFT JOIN + COALESCE: customer-level features без множення review
----------------------------------------------------------------------------------------------------


,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


----------------------------------------------------------------------------------------------------
4.3. FULL OUTER JOIN: які штати присутні серед клієнтів і продавців
----------------------------------------------------------------------------------------------------


,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


----------------------------------------------------------------------------------------------------
4.4. SELF JOIN: пари продавців з одного штату, але з різних міст
----------------------------------------------------------------------------------------------------


,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


----------------------------------------------------------------------------------------------------
4.5. Anti-join: замовлення без відгуку
----------------------------------------------------------------------------------------------------


,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


### 5. GROUP BY, HAVING і агрегати

In [16]:
SQL_1 = text("""
    SELECT
        p.product_category_name,
        SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue
    FROM olist_order_items AS oi
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY p.product_category_name
    HAVING SUM(oi.price) > 10000
    ORDER BY gross_revenue DESC NULLS LAST;
""")

SQL_2 = text("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            c.customer_state,
            SUM(oi.price + oi.freight_value) AS order_total
        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
        JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            c.customer_state
    )
    SELECT
        customer_state,
        AVG(order_total)::NUMERIC(8, 2) AS aov
    FROM order_totals
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
    ORDER BY aov DESC NULLS LAST;
    """)

SQL_3 = text("""
    SELECT
        c.customer_unique_id,
        SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
        STRING_AGG(
            DISTINCT COALESCE(p.product_category_name, '(unknown)'),
            ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
        ) AS categories
    FROM olist_customers AS c
    JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY c.customer_unique_id
    HAVING COUNT(DISTINCT o.order_id) >= 2
    ORDER BY total_spend DESC NULLS LAST
    LIMIT 20;
    """)

queries = [
    {'title': '1. Revenue per product category', 'stmt': SQL_1},
    {'title': '2. AOV per customer state', 'stmt': SQL_2},
    {'title': '3. Категорії повторних покупців (STRING_AGG)', 'stmt': SQL_3},
]

for query in queries:
    with engine.begin() as conn:
        res = conn.execute(query['stmt'])
    res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
    print("-"*100)
    print(query['title'])
    print("-"*100)
    display(res_df)

----------------------------------------------------------------------------------------------------
1. Revenue per product category
----------------------------------------------------------------------------------------------------


,product_category_name,gross_revenue
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


----------------------------------------------------------------------------------------------------
2. AOV per customer state
----------------------------------------------------------------------------------------------------


,customer_state,aov
0,PB,265.01
1,AL,234.13
2,RO,233.03
3,PA,224.38
4,TO,219.91
5,PI,219.34
6,SE,211.69
7,RN,211.40
8,CE,207.69
9,MT,206.17


----------------------------------------------------------------------------------------------------
3. Категорії повторних покупців (STRING_AGG)
----------------------------------------------------------------------------------------------------


,customer_unique_id,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2009.98,consoles_games


**SQL_1** - `gross_revenue` - виручка категорії без доставки. `HAVING` залишає лише категорії з виручкою понад `10 000`.

**SQL_2** - `aov` (Average Order Value - середній чек). `HAVING` залишає лише штати з мінімум `100` замовленнями, щоб середнє було надійним.

**SQL_3** - `total_spend` - сума покупок клієнта без доставки. `HAVING` залишає лише повторних покупців (≥ 2 замовлення), `categories` показує, що саме вони купували.

### `WHERE` vs `HAVING`

**Різниця:**
`WHERE` фільтрує окреми рядки, тоді як `HAVING` фільтрує групи і використовується після `GROUP BY` та може виконувати агрегатні функуції накшалт `SUM`, etc.

## 6. Set operations, підзапит/window і Reflection

EXCEPT EXAMPLE - orders without reviews

In [17]:
SQL = text("""
SELECT order_id
FROM olist_orders

EXCEPT

SELECT order_id
FROM olist_order_reviews;
""")

with engine.begin() as conn:
    res = conn.execute(SQL)
res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,order_id
0,d30a31ae450b7eccbb5fc54276bdccf7
1,a111f1d17afc3e76dd0c2428a96a7617
2,0bd6f6cbd1aa613a762bcc714ffcadd5
3,19428d94d47ebaa715faa60f63186de6
4,bd1befc72cade5c00562b4bfd1a85485
...,...
763,e349cfe864fffc00009b4cdbdd3a95ec
764,2c2ad0e6636d842e7f1eead45cdc6065
765,d2edbdf7b05aec925d270f089e79443e
766,82172f2c466d4e1298dab08c8f0e111c


### Найновіший review кожного реального покупця

Review прив'язаний до замовлення, тому через `olist_orders` і `olist_customers` кожен review з'єднується з `customer_unique_id` (реальною людиною).

`ROW_NUMBER()` нумерує review окремо для кожного покупця (`PARTITION BY customer_unique_id`), починаючи з найновішого (`ORDER BY review_creation_date DESC`). `review_id DESC` - додаткове сортування на випадок, коли в покупця кілька review з однаковою датою.

`WHERE rn = 1` залишає лише один, найновіший review на покупця.

In [18]:
SQL = text("""
WITH ranked_reviews AS (
    SELECT
        c.customer_unique_id,
        rv.review_id,
        rv.order_id,
        rv.review_score,
        rv.review_creation_date,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY rv.review_creation_date DESC, rv.review_id DESC
        ) AS rn
    FROM olist_order_reviews AS rv
    JOIN olist_orders AS o
        ON o.order_id = rv.order_id
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    review_id,
    order_id,
    review_score,
    review_creation_date
FROM ranked_reviews
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

with engine.begin() as conn:
    res = conn.execute(SQL)
res_df = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
display(res_df)

,customer_unique_id,review_id,order_id,review_score,review_creation_date
0,0000366f3b9a7992bf8c76cfdf3221e2,13359f48c3c7e41a2b8afca340937eeb,e22acc9c116caa3f2b7121bbb380d08e,5,2018-05-17
1,0000b849f77a49e4a4ce2b2a4ca5be3f,e383e10e36a765f04f752a37c1674e43,3594e05a005ac4d06a72673270ef9ec9,4,2018-05-11
2,0000f46a3911fa3c0805444483337064,d89cb4217c23008b141e66c3160de8a4,b33ec3b699337181488304f362a6b734,3,2017-04-06
3,0000f6ccb0745a6a4b88665a16c9f078,6f22e4ad2cd9e0d40c5842e4787b4321,41272756ecddd9a9ed0180413cc22fb6,4,2017-11-02
4,0004aac84e0df4da2b147fca70cf8255,880a77e7dd4e496969f138f974cc44a1,d957021f1127559cd947b62533f484f7,5,2017-11-28
5,0004bd2a26a76fe21f786e4fbd80607f,09f0931c60418e8c0ba6cac5658a7b28,3e470077b690ea3e3d501cffb5e0c499,4,2018-04-08
6,00050ab1314c0e55a6ca13cf7181fecf,ffc1a4080d67b1d3d06749f4f2d3ee59,d0028facea13f508e880202d7097a5a1,4,2018-04-28
7,00053a61a98854899e70ed204dd4bafe,333a6ca3cdb01f5f00e3e6be1f1236e1,44e608f2db00c74a1fe329de44416a4e,1,2018-03-17
8,0005e1862207bf6ccc02e4228effd9a0,6b298d209aa5d83663c18cf6fa0be2ff,ae76bef74b97bcb0b3e355e60d9a6f9c,4,2017-03-10
9,0005ef4cd20d2893f0d9fbd94d3c0d97,df4e1e05fa40b3fff7574e386f904afd,01b330808c5819a6a3cb79b72f0b8288,1,2018-04-06


## Reflection

**1. Який тип `JOIN` був найскладнішим і чому?**

`LEFT JOIN`. Його легко написати, але він додає `NULL` у рядки без пари, і це далі впливає на `AVG`, `COUNT` і `CASE`. Наприклад, у `seller_score` без окремої перевірки на `NULL` продавці без review отримали б tier `bronze`.

**2. Який запит найскладніше було сформулювати як business-question?**

Запит з `ROW_NUMBER()`: треба було чітко визначити, що означає "останній review покупця", і врахувати, що в одного покупця може бути кілька review з однаковою датою.

**3. Яка PostgreSQL-фіча з Теми 4 здається найбільш корисною для production-ETL: `RETURNING`, `ON CONFLICT`, identity-ключі, constraints або window functions?**

`ON CONFLICT`. Він дозволяє запускати той самий скрипт повторно без дублікатів - дані просто оновлюються.

**4. У яких місцях такого e-commerce pipeline ви б використали window functions у реальному ML-проєкті?**

Останнє замовлення або review клієнта.

**5. Які data-quality проблеми ви помітили в Olist-таблицях: відсутні reviews, пропущені категорії, дивні статуси, дублікати або порожні тексти?**

Замовлення без review, товари без категорії, скасовані й недоставлені замовлення поруч із доставленими, кілька review на одне замовлення, порожні тексти коментарів. Також `review_creation_date` містить лише дату без часу.

**6. Чому для customer-level ML-фіч у цьому ДЗ використано `customer_unique_id`, а не `customer_id`?**

`customer_id` в Olist створюється для кожного замовлення, а `customer_unique_id` - це реальна людина. Без нього кожен клієнт виглядав би як покупець з одним замовленням.

**7. Чому перед обчисленням `seller_score` потрібно було привести дані до грануляності `seller_id × order_id`?**

`olist_order_items` має рядок на кожен товар, а review ставиться на замовлення. Без цього кроку замовлення з кількома товарами рахувалось би кілька разів і отримувало б більшу вагу в рейтингу.